# Mosaic eval harness — GPU run on Colab

Runs `src/eval/{shortcut_check,robustness,calibration,error_analysis}.py` with `--device cuda`, against CIFAKE + SID-Set + WildFake combined (via `scan_all_sources()` in `src/eval/data_compat.py`).

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU.

**What you need on hand:**
- `mosaic_code.zip` (src/, configs/, requirements.txt, pyproject.toml, plus the already-cleaned SID-Set/WildFake manifests — no raw images, no checkpoint)
- Your checkpoint `.pt` file (e.g. `wildfake_head.pt`, or `model_best.pt`)
- A Kaggle API token (`kaggle.json`) to fetch CIFAKE — from https://www.kaggle.com/settings -> API -> Create New Token
- **For WildFake**: open https://modelscope.cn/datasets/hy2628982280/WildFake/summary and click "translate" once, per the challenge instructions — this can't be scripted. Download `church.zip`, `DDIM.zip`, `DDPM.zip` (~8GB each) and upload them to a folder named `wildfake` in your Google Drive (`My Drive/wildfake/`) before running the notebook's WildFake cell — it copies them from there rather than through a browser upload button.

## 1. Upload the code

In [ ]:
from google.colab import files

print("Select mosaic_code.zip")
uploaded = files.upload()
assert any(name.endswith(".zip") for name in uploaded), "Expected a .zip file"
zip_name = [name for name in uploaded if name.endswith(".zip")][0]

In [ ]:
import shutil

shutil.unpack_archive(zip_name, "/content/mosaic")
%cd /content/mosaic
!ls

## 2. Install dependencies

Colab's preinstalled torch already includes CUDA support for Linux, so a normal `pip install -r requirements.txt` picks up a CUDA-enabled torch build without any special index URL. Then install the project itself in editable mode so `import src.xxx` resolves regardless of invocation style (mirrors the local `pip install -e .` fix).

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -20
!pip install -q -e . 2>&1 | tail -20

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — check Runtime > Change runtime type > GPU")

## 3. Upload the checkpoint

In [ ]:
import os
os.makedirs("outputs/baseline", exist_ok=True)

print("Select your checkpoint .pt file")
uploaded = files.upload()
ckpt_name = list(uploaded)[0]
shutil.move(ckpt_name, "outputs/baseline/model_best.pt")
!ls -la outputs/baseline/

## 4a. Fetch CIFAKE (downloaded fresh here — faster than uploading 469MB from a laptop)

Reproduces the same `data/raw/CIFAKE/{train,test}/{REAL,FAKE}` layout used locally (native Kaggle shape, matching `src/eval/data_compat.py`'s `scan_cifake_nested`).

In [ ]:
# --- Option A: paste your Kaggle username/key directly (use this if the
# kaggle.json download from kaggle.com/settings didn't work in your browser —
# get these two values from your browser's DevTools > Network tab instead,
# see the kaggle.json response for the "Create New Token" request) ---
import json
import os

KAGGLE_USERNAME = "PASTE_YOUR_USERNAME_HERE"
KAGGLE_KEY = "PASTE_YOUR_KEY_HERE"

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# --- Option B: upload the kaggle.json file instead (uncomment if you have it) ---
# print("Select your kaggle.json API token")
# uploaded = files.upload()
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
import kagglehub

cifake_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print("Downloaded to:", cifake_path)
!ls "{cifake_path}"

In [ ]:
# Symlink kagglehub's cache into data/raw/CIFAKE/{train,test} — same trick used locally,
# avoids copying ~120K files.
os.makedirs("data/raw/CIFAKE", exist_ok=True)
for split in ("train", "test"):
    src = os.path.join(cifake_path, split)
    dst = f"data/raw/CIFAKE/{split}"
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
!find data/raw/CIFAKE -maxdepth 2
!find data/raw/CIFAKE -type f | wc -l

## 4b. Fetch SID-Set (a small subset — the full dataset is ~141GB)

SID-Set's full Parquet dataset is about 141GB (249 train shards + 34 validation shards) — bigger than a standard Colab disk on its own, which is what caused the "disk almost full" error. Instead of downloading everything (`snapshot_download`), this pulls just a handful of individual shard files. `src/eval/data_compat.py`'s scanners automatically skip any manifest row whose shard isn't downloaded (not an error) — so this gives a smaller but real, working SID-Set sample. Raise `N_TRAIN_SHARDS`/`N_VAL_SHARDS` later if you have disk room to spare.

In [ ]:
from huggingface_hub import hf_hub_download

N_TRAIN_SHARDS = 5   # of 249 total — each averages roughly 500MB
N_VAL_SHARDS = 2     # of 34 total

shard_names = (
    [f"data/train-{i:05d}-of-00249.parquet" for i in range(N_TRAIN_SHARDS)]
    + [f"data/validation-{i:05d}-of-00034.parquet" for i in range(N_VAL_SHARDS)]
)
for shard in shard_names:
    hf_hub_download(repo_id="saberzl/SID_Set", repo_type="dataset", filename=shard, local_dir="data/raw/sid_set")

!du -sh data/raw/sid_set
!find data/raw/sid_set -name "*.parquet"

## 4c. Fetch WildFake (manual step required first)

ModelScope's translate-page requirement means this can't be automated — after translating the page, download `church.zip`, `DDIM.zip`, `DDPM.zip` (the 3 you're using) and upload them to your Google Drive first (drag-and-drop on drive.google.com, in your own time — handles multi-GB files far better than a browser upload button). Then the cell below mounts your Drive and copies them in.

Archive paths must match `clean_wildfake.py`'s `DEFAULT_ARCHIVES` exactly, relative to `data/raw/wildfake/`:
`Images/Real/church.zip`, `Images/Diffusion_based/{DDIM,DDPM}.zip`.

In [ ]:
import os

os.makedirs("data/raw/wildfake/Images/Real", exist_ok=True)
os.makedirs("data/raw/wildfake/Images/Diffusion_based", exist_ok=True)

# Matches filenames case-insensitively and by substring (e.g. "Church (1).zip"
# or "church_v2.zip" both still match "church") rather than requiring an
# exact name — a browser-renamed duplicate download shouldn't silently fail
# to place a file, which the earlier exact-match version was prone to.
REAL_KEYWORDS = ("church", "afhq", "celebahq")
FAKE_KEYWORDS = ("ddim", "ddpm")


def route(name: str) -> str | None:
    lower = name.lower()
    if any(k in lower for k in REAL_KEYWORDS):
        return f"data/raw/wildfake/Images/Real/{name}"
    if any(k in lower for k in FAKE_KEYWORDS):
        return f"data/raw/wildfake/Images/Diffusion_based/{name}"
    return None


# --- Option A: Google Drive (recommended here — church.zip/DDIM.zip/DDPM.zip
# are ~8GB each; upload them to your Drive first, then this copies them in) ---
from google.colab import drive

drive.mount("/content/drive")

WILDFAKE_DRIVE_DIR = "/content/drive/MyDrive/wildfake"  # change if you used a different folder name
for name in os.listdir(WILDFAKE_DRIVE_DIR):
    dest = route(name)
    if dest is None:
        print(f"Skipping {name} — doesn't match any known archive name")
        continue
    shutil.copy(os.path.join(WILDFAKE_DRIVE_DIR, name), dest)
    print(f"Copied {name} -> {dest}")

# --- Option B: direct browser upload instead (uncomment if you'd rather not use Drive) ---
# print("Select your WildFake archive(s)")
# uploaded = files.upload()
# for name in uploaded:
#     dest = route(name)
#     if dest is None:
#         print(f"Skipping {name} — doesn't match any known archive name")
#         continue
#     shutil.move(name, dest)

!find data/raw/wildfake -maxdepth 3

## 5. Run the eval scripts on GPU

Each accepts `--device` — pass `cuda` here instead of the `cpu` default used locally. `shortcut_check.py` first, per the project's priority order (early warning on shortcut learning before polishing the rest).

In [ ]:
!python src/eval/shortcut_check.py --checkpoint outputs/baseline/model_best.pt --device cuda

In [ ]:
!python src/eval/robustness.py --checkpoint outputs/baseline/model_best.pt --device cuda --out outputs/robustness_table.csv

In [ ]:
!python src/eval/calibration.py --checkpoint outputs/baseline/model_best.pt --device cuda

In [ ]:
!python src/eval/error_analysis.py --checkpoint outputs/baseline/model_best.pt --device cuda --out outputs/error_analysis.md

## 6. Download the results back to your laptop

In [ ]:
shutil.make_archive("/content/mosaic_outputs", "zip", "outputs")
files.download("/content/mosaic_outputs.zip")